# Service Kernel Gateway — CFR OpenSpiel

Service du **pli 0** de l'Epic #19571 : exposer le carnet phare
[GameTheory-13-ImperfectInfo-CFR-Python.ipynb](../GameTheory-13-ImperfectInfo-CFR-Python.ipynb)
en microservice REST via Jupyter **Kernel Gateway** (mode `notebook-http`).

**Contrat** : `POST /gametheory/cfr` — corps `{"game": "kuhn_poker", "iterations": 300}`
-> `{"status": "ok", "game": ..., "iterations": ..., "exploitability": ..., "nash_conv": ..., "policy": {...}}`.

Jeux supportes : `kuhn_poker`, `leduc_poker`, `matrix_rps` (chargé via la transformation
`turn_based_simultaneous_game`, CFR exigeant un arbre séquentiel).

Ce carnet s'exécute aussi **hors gateway** (mode local) : il sert alors de démonstration
sur `kuhn_poker`. Voir le [README](README.md) pour le démarrage.

Le carnet se lit en trois temps : la **détection du mode** (le même code sert le HTTP
et la démonstration locale), l'**endpoint annoté** (`# POST /gametheory/cfr`, exécuté à
chaque requête), puis la **lecture de la réponse** — l'équilibre approché du poker de
Kuhn calculé par minimisation du regret contrefactuel, lu valeur par valeur.

In [1]:
# Cellule de preparation — executee une fois au demarrage du kernel.
import json
import os
import sys

# Le kernel demarre dans ce dossier (seed_uri) : il y trouve openspiel_cfr_core.py.
sys.path.insert(0, os.getcwd())

from openspiel_cfr_core import solve_cfr

# REQUEST n'existe que sous Kernel Gateway (mode notebook-http) ; l'objet est
# injecte au moment de chaque requete. Hors gateway : None -> mode local.
REQUEST = globals().get("REQUEST")

# Jeu de demonstration (carnet phare GameTheory-13 : poker de Kuhn).
DEMO_GAME = "kuhn_poker"
DEMO_ITERATIONS = 300

print("Mode :", "Kernel Gateway (notebook-http)" if REQUEST is not None else "local (demonstration)")


Mode : local (demonstration)


### Lecture du mode — un même carnet, deux exécutions

La sortie dit `Mode : local (demonstration)` : `REQUEST` est absent de l'espace de
noms, le kernel n'a donc pas été lancé par Kernel Gateway. Sous gateway, la même
cellule afficherait `Kernel Gateway (notebook-http)` — c'est ce test de présence, et
non un paramètre de configuration, qui bascule le carnet entier d'un mode à l'autre.

Deux détails de préparation se lisent aussi ici. `sys.path.insert(0, os.getcwd())` :
l'import d'`openspiel_cfr_core` qui suit réussit, donc le dossier courant est bien
celui du carnet — le service n'installe rien. Le réglage qui place le kernel dans ce
dossier (`seed_uri`) appartient au lanceur Kernel Gateway, documenté dans le README
du dossier `gateway/` : c'est une donnée de déploiement, pas une mesure de cette
exécution. Enfin `DEMO_ITERATIONS = 300` fixe le budget de la démonstration locale :
300 itérations, assez pour que l'exploitabilité descende sous le centième — la sortie
du endpoint la montrera à 0,003.

## Endpoint `POST /gametheory/cfr`

La cellule ci-dessous est annotée en première ligne par `# POST /gametheory/cfr` :
Kernel Gateway l'exécute à chaque requête HTTP, avec l'objet `REQUEST` injecté
dans l'espace de noms. Le JSON imprimé devient le corps de la réponse.

La cellule absorbe trois formats d'entrée : `REQUEST` en chaîne JSON (Kernel Gateway
v3, où le serveur double le `json.dumps`), `REQUEST` en dict (versions antérieures),
et l'absence totale de `REQUEST` — le mode local, où la paire `DEMO_GAME` /
`DEMO_ITERATIONS` tient lieu de requête. Le calcul lui-même est délégué à
`solve_cfr` : tout le domaine (OpenSpiel, la boucle CFR, l'agrégation de la
politique moyenne) vit dans le module du carnet phare, le service n'est qu'une
enveloppe HTTP au-dessus — modifier l'algorithme ne touche jamais ce carnet.

In [2]:
# POST /gametheory/cfr
# Sous Kernel Gateway, REQUEST porte la requete : chaine JSON en v3
# (double json.dumps cote serveur), dict direct selon les versions -- on
# accepte les deux. Le corps JSON (Content-Type application/json) est
# deja parse dans la cle "body".
requete = None
if isinstance(REQUEST, str):
    try:
        requete = json.loads(REQUEST)
    except Exception:
        requete = None
elif isinstance(REQUEST, dict):
    requete = REQUEST
corps = requete.get("body") if isinstance(requete, dict) else None
game = corps.get("game") if isinstance(corps, dict) else None
iterations = corps.get("iterations") if isinstance(corps, dict) else None
if REQUEST is None:
    # Execution locale : demonstration sur le jeu du carnet phare.
    game = DEMO_GAME
    iterations = DEMO_ITERATIONS
reponse = solve_cfr(game, iterations)
print(json.dumps(reponse))


{"status": "ok", "game": "kuhn_poker", "iterations": 300, "exploitability": 0.003034, "nash_conv": 0.006068, "policy": {"0": {"Pass": 0.811179, "Bet": 0.188821}, "1p": {"Pass": 0.988333, "Bet": 0.011667}, "0pb": {"Pass": 0.998973, "Bet": 0.001027}, "1b": {"Pass": 0.658747, "Bet": 0.341253}, "2p": {"Pass": 0.003333, "Bet": 0.996667}, "2b": {"Pass": 0.001667, "Bet": 0.998333}, "1": {"Pass": 0.975026, "Bet": 0.024974}, "0p": {"Pass": 0.665731, "Bet": 0.334269}, "1pb": {"Pass": 0.470903, "Bet": 0.529097}, "0b": {"Pass": 0.998333, "Bet": 0.001667}, "2": {"Pass": 0.431393, "Bet": 0.568607}, "2pb": {"Pass": 0.001932, "Bet": 0.998068}}}


### Lecture de la réponse — l'équilibre du poker de Kuhn, valeur par valeur

Le JSON ci-dessus est la réponse complète pour 300 itérations. Trois familles
d'informations s'y lisent.

**Convergence.** `exploitability: 0.003034` : en moyenne sur les deux joueurs, se
détourner de la politique rendue ne rapporterait que 0,003 pot par partie — après
300 itérations, CFR a pressé l'écart à l'équilibre sous le tiers du centième.
`nash_conv: 0.006068` en est exactement le double (somme des regrets d'exploitation
des deux joueurs au lieu de leur moyenne) : les deux mesures racontent la même
convergence, et leur ratio de 2 se vérifie chiffre à chiffre dans la sortie.

**Structure.** `policy` compte 12 états d'information. Les codes sont
`<carte><histoire>` — carte 0/1/2 = Valet/Dame/Roi, histoire en `p`/`b` (passe/mise,
et `b` en seconde position signifie suivre). Chaque état porte une loi sur
`{Pass, Bet}` qui somme à 1 (`0` : 0,811 + 0,189 = 1).

**L'équilibre, là où il est tranché.** Les valeurs extrêmes disent les stratégies
pures : le Roi mise après une passe (`2p` : Bet 0,997) et paie toujours (`2b`, `2pb` :
Bet ≈ 0,998) ; le Valet se couche face à une mise après avoir passé lui-même
(`0pb` : Bet 0,001). Les valeurs intermédiaires disent les mélanges d'équilibre :
le Valet bluffe après une passe un tiers du temps (`0p` : Bet 0,334 — la fréquence
analytique du bluff au poker de Kuhn), la Dame paie une mise après avoir passé dans
53 % des cas (`1pb` : Bet 0,529 — le rôle de bluff-catcher), et ne suit l'ouverture
que dans 34 % des cas (`1b` : Bet 0,341). C'est la signature d'un équilibre mixte :
maximiser en mélangeant juste assez pour laisser l'adversaire indifférent entre ses
options.

## Ce que ce service démontre

Le carnet phare prouve que CFR converge ; ce service prouve que le résultat est
**consommable** : un `POST` avec deux paramètres rend une politique jouable, son
exploitabilité et sa convergence — sans notebook ni OpenSpiel côté client. La
démonstration locale montre l'autre moitié du pli : le même code sert le HTTP et la
pédagogie, la bascule étant un simple test de présence de `REQUEST`.